# Biomarker S10 — 96 ca holdout cùng nguồn OAI-ZIB: audit và chuẩn bị

**Kế hoạch:** `docs/ke_hoach_m3t_cls.md` mục 5b. Sản phẩm ghi vào `.../knee_biomarkers_09_09/s10_oaizib_holdout/`,
**không** ghi thêm vào bảng v2 1229 ca.

Tên `external` chỉ giữ cho tương thích manifest: 96 ca này là **holdout cùng nguồn OAI-ZIB**, không phải dữ liệu
ngoài độc lập. Notebook này mới làm **bước 1** của mục 5b:

1. **Ghim manifest 96 ca** — đếm KL, rời subject với 1229 ca phát triển, ảnh tồn tại.
2. **Audit mô hình segmentation** đã sinh mask 09_09: danh sách train/val của mọi fold, hash checkpoint, và kiểm
   trùng **nội dung ảnh** (dấu vân tay) lẫn **subject** giữa 96 ca và dữ liệu train của mô hình đó.

Có phơi nhiễm hoặc thiếu provenance → **chưa được gọi là đánh giá trên ca chưa thấy**; phải có checkpoint sạch trước
khi chạy các bước sau (segmentation 96 ảnh, S3/S6, ghép CLS, đánh giá một lần) — các bước đó chưa triển khai ở đây.

## 0) Môi trường

In [ ]:
!pip install -q openpyxl 2>/dev/null
from google.colab import drive
drive.mount("/content/drive")

REPO_URL, REPO_DIR = "https://github.com/AIVIETNAM-AIO-Tuan/bsCart-net.git", "/content/repo"
import os, sys
if not os.path.isdir(f"{REPO_DIR}/bsc"):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
for _m in [k for k in list(sys.modules) if k == "bsc" or k.startswith("bsc.")]:
    del sys.modules[_m]

import hashlib, json, re
from pathlib import Path
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from bsc import io_utils as IO, m3t as M3T, m3t_train as T, track

GIT_SHA = track.git_sha(REPO_DIR)
BIOM_DIR = Path("/content/drive/MyDrive/OAI_seg/knee_biomarkers_09_09")
OUT = BIOM_DIR / "s10_oaizib_holdout"
OUT.mkdir(parents=True, exist_ok=True)
IO.assert_drive_first(OUT)
MANIFEST = BIOM_DIR / "cohort_manifest.csv"
V2_CSV = BIOM_DIR / "s6_fcl" / "biomarker_table_v2.csv"
PATH_REMAPS = [("/content/drive/MyDrive/nnUNet_raw", "/content/drive/MyDrive/OAI_seg/nnUNet_raw"),
               ("/content/drive/MyDrive/OAI_DESS", "/content/drive/MyDrive/OAI_seg/OAI_DESS")]   # <<< sua neu Drive khac
# Doi chieu cua nguoi dung ngay 25/09/2026 (ke hoach muc "Su that da kiem"): KL0..4 cua 96 ca
EXPECTED_KL = {0: 21, 1: 12, 2: 22, 3: 26, 4: 15}
print(f"git {GIT_SHA[:12]} | da co trong {OUT.name}/:", sorted(p.name for p in OUT.iterdir()))

## 1) Ghim manifest 96 ca

Lấy từ `cohort_manifest.csv` các ca `source_dataset = reserved_external_validation_*` (hoặc `oaizib_split` chứa `ext`).
Cổng: đúng **96 ca**, KL0–4 = **21/12/22/26/15**, **không subject nào** trùng 1229 ca phát triển, **đủ ảnh**.
Bên gối/lần khám của OAI-ZIB do S1 gán (`R`, `V00`) — chưa kiểm ở đây; S9 mục 6 xác minh bên gối bằng ảnh cho ca có npz.

In [ ]:
MAN = pd.read_csv(MANIFEST).drop_duplicates("case_id").reset_index(drop=True)
_is_ext = MAN.source_dataset.astype(str).str.startswith("reserved_external_validation")
if "oaizib_split" in MAN.columns:
    _is_ext |= MAN.oaizib_split.astype(str).str.contains("ext", case=False, na=False)
HO = MAN[_is_ext].copy().reset_index(drop=True)
DEV = MAN[~_is_ext]
V2 = pd.read_csv(V2_CSV, usecols=["case_id", "subject"])
print(f"holdout {len(HO)} | phat trien trong manifest {len(DEV)} | bang v2 {len(V2)}")
print("nguon:", HO.source_dataset.value_counts().to_dict())
_kl = {int(k): int(v) for k, v in HO.KL.astype(int).value_counts().sort_index().items()}
print("KL holdout:", _kl, "| doi chieu:", EXPECTED_KL)

HO["subject_n"] = HO.subject.map(T.norm_subject)
_dev_subj = set(DEV.subject.map(T.norm_subject)) | set(V2.subject.map(T.norm_subject))
_common = sorted(set(HO.subject_n) & _dev_subj)
HO["dess_resolved"] = [IO.resolve_path(p, PATH_REMAPS, must_exist=False) for p in HO.dess_path.astype(str)]
HO["img_family"] = HO.dess_path.map(T.image_family)
HO["img_name"] = HO.dess_path.astype(str).str.replace("\\", "/", regex=False).str.split("/").str[-1]
HO["img_split"] = HO.dess_path.astype(str).str.extract(r"(imagesTr|imagesTs)")[0].fillna("?")
print("\nho anh x thu muc goc:\n", pd.crosstab(HO.img_family, HO.img_split))
T.write_once_csv(HO, OUT / "holdout_manifest.csv")
if not (OUT / "holdout_manifest.json").exists():          # co git SHA -> chi ghi lan dau
    T.write_once_json(dict(n=len(HO), kl=_kl, expected_kl=EXPECTED_KL, n_subjects=HO.subject_n.nunique(),
                           subjects_shared_with_dev=_common, n_missing_image=int(HO.dess_resolved.isna().sum()),
                           img_split=HO.img_split.value_counts().to_dict(), git=GIT_SHA), OUT / "holdout_manifest.json")

assert len(HO) == 96, f"{len(HO)} ca, can 96"
assert not set(HO.case_id) & set(V2.case_id), "ca holdout nam trong bang v2"
assert _kl == EXPECTED_KL, f"KL holdout {_kl} != {EXPECTED_KL} - DUNG, bao Claude"
assert not _common, f"{len(_common)} subject holdout trung cohort phat trien: {_common[:5]}"
assert HO.dess_resolved.notna().all(), f"{int(HO.dess_resolved.isna().sum())} ca thieu anh"
print("manifest 96 ca: DAT")

## 2) Audit mô hình segmentation đã sinh mask 09_09

**Phải điền đúng mô hình thực sự dùng.** S2 (cũ) dùng `Dataset020_KneeUnion`, ResEnc-L 250 epoch, **fold 0**,
`checkpoint_final.pth`; mask của bảng 09_09 được dựng ngoài repo nên đây là **giả định cần bạn xác nhận** — sửa
`SEG_*` nếu khác (kể cả khi là ensemble nhiều fold).

Kiểm ba lớp:
1. **Provenance:** checkpoint tồn tại, sha256, `splits_final.json` (train/val của từng fold).
2. **Trùng nội dung ảnh:** dấu vân tay mọi ảnh `imagesTr` của dataset train mô hình với 96 ảnh holdout
   (cùng file → r ≈ 1; ngưỡng 0,99).
3. **Trùng subject:** tên ca của dataset train → subject (số 7 chữ số trong tên, hoặc `oaizib_XXX` qua `subInfo`),
   so với subject của 96 ca — bắt cả gối kia, lần khám khác.

In [ ]:
def _first(*cands):
    return next((Path(c) for c in cands if Path(c).exists()), None)


SEG_DATASET = "Dataset020_KneeUnion"                                           # <<< dataset train mo hinh seg
SEG_TRAINER_DIR = "nnUNetTrainer_250epochs__nnUNetResEncUNetLPlans__3d_fullres"   # <<< trainer__plans__config
SEG_FOLDS = [0]                                                                # <<< MOI fold dung de sinh mask
SEG_CHECKPOINT = "checkpoint_final.pth"                                        # <<< checkpoint da dung
SEG_RESULTS = _first("/content/drive/MyDrive/OAI_seg/nnUNet_results", "/content/drive/MyDrive/nnUNet_results")
SEG_PREP = _first("/content/drive/MyDrive/OAI_seg/nnUNet_preprocessed", "/content/drive/MyDrive/nnUNet_preprocessed")
SEG_RAW = _first("/content/drive/MyDrive/OAI_seg/nnUNet_raw", "/content/drive/MyDrive/nnUNet_raw")
SUBINFO = [p for p in (_first("/content/drive/MyDrive/subInfo_train.xlsx", "/content/drive/MyDrive/OAI_seg/subInfo_train.xlsx"),
                       _first("/content/drive/MyDrive/subInfo_test.xlsx", "/content/drive/MyDrive/OAI_seg/subInfo_test.xlsx"))
           if p is not None]
print("results:", SEG_RESULTS, "| preprocessed:", SEG_PREP, "| raw:", SEG_RAW, "| subInfo:", SUBINFO)

# ---- 2.1 provenance: checkpoint + splits
PROV = dict(dataset=SEG_DATASET, trainer_dir=SEG_TRAINER_DIR, folds=SEG_FOLDS, checkpoint=SEG_CHECKPOINT,
            checkpoints={}, splits_found=False, missing=[])
_mdir = SEG_RESULTS / SEG_DATASET / SEG_TRAINER_DIR if SEG_RESULTS else None


def _sha256(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


for f in SEG_FOLDS:
    ck = _mdir / f"fold_{f}" / SEG_CHECKPOINT if _mdir else None
    if ck is not None and ck.exists():
        PROV["checkpoints"][f"fold_{f}"] = dict(path=str(ck), sha256=_sha256(ck), bytes=ck.stat().st_size)
    else:
        PROV["missing"].append(f"checkpoint fold_{f}")
_splits_p = _first(*(p for p in [SEG_PREP / SEG_DATASET / "splits_final.json" if SEG_PREP else None,
                                 _mdir / "splits_final.json" if _mdir else None] if p is not None))
SPLITS = json.load(open(_splits_p)) if _splits_p else None
PROV["splits_found"] = SPLITS is not None
if SPLITS is None:
    PROV["missing"].append("splits_final.json")
ROLE = {}                                                    # ten ca -> 'train' / 'val' trong fold da dung
for f in SEG_FOLDS:
    if SPLITS is not None and f < len(SPLITS):
        for c in SPLITS[f]["train"]:
            ROLE.setdefault(c, "train")
        for c in SPLITS[f]["val"]:
            ROLE[c] = "train" if ROLE.get(c) == "train" else "val"
_img_dir = SEG_RAW / SEG_DATASET / "imagesTr" if SEG_RAW else None
TRAIN_IMGS = sorted(_img_dir.glob("*_0000.nii.gz")) if _img_dir and _img_dir.exists() else []
if not TRAIN_IMGS:
    PROV["missing"].append(f"{SEG_DATASET}/imagesTr")
print(f"checkpoint: {list(PROV['checkpoints'])} | splits: {PROV['splits_found']} | anh imagesTr: {len(TRAIN_IMGS)} | "
      f"THIEU: {PROV['missing']}")

In [ ]:
# ---- 2.2 trung NOI DUNG anh: dau van tay imagesTr cua dataset train vs 96 anh holdout
FP_CSV = OUT / "seg_train_fingerprints.csv"
SAME_IMG_R = 0.99
if FP_CSV.exists():
    _fp_train = pd.read_csv(FP_CSV)
    BANK = _fp_train[[c for c in _fp_train.columns if c.startswith("fp_")]].to_numpy(np.float32)
    TRAIN_NAMES = _fp_train.case.tolist()
else:
    TRAIN_NAMES = [p.name[: -len("_0000.nii.gz")] for p in TRAIN_IMGS]
    BANK = np.stack([M3T.fingerprint(IO.load_nii(str(p))[0]) for p in tqdm(TRAIN_IMGS, desc="imagesTr")]) \
        if TRAIN_IMGS else np.zeros((0, int(np.prod(M3T.FP_SHAPE))), np.float32)
    _fp_train = pd.DataFrame(BANK, columns=[f"fp_{i}" for i in range(BANK.shape[1])])
    _fp_train.insert(0, "case", TRAIN_NAMES)
    T.write_once_csv(_fp_train, FP_CSV)

# ---- 2.3 trung SUBJECT: ten ca -> subject (so 7 chu so, hoac oaizib_XXX qua subInfo)
CMT2SUBJ = {}
for p in SUBINFO:
    _si = pd.read_excel(p)
    CMT2SUBJ.update(dict(zip(_si["CMT-ID"].astype(str).str.zfill(3), _si["SubjectID"].map(T.norm_subject))))


def subject_of(name):
    m = re.search(r"(?<!\d)(9\d{6})(?!\d)", name)
    if m:
        return m.group(1)
    m = re.search(r"oaizib_(\d+)", name)
    return CMT2SUBJ.get(m.group(1).zfill(3)) if m else None


TRAIN_SUBJ = {n: subject_of(n) for n in TRAIN_NAMES}
print(f"ten ca train map duoc subject: {sum(v is not None for v in TRAIN_SUBJ.values())}/{len(TRAIN_SUBJ)}")

_rows = []
for r in tqdm(list(HO.itertuples()), desc="holdout"):
    fp = M3T.fingerprint(IO.load_nii(r.dess_resolved)[0])
    rmax, i = M3T.max_corr(fp, BANK) if len(BANK) else (np.nan, -1)
    match = TRAIN_NAMES[i] if i >= 0 else None
    same_subj = [n for n, s in TRAIN_SUBJ.items() if s is not None and s == r.subject_n]
    _rows.append(dict(case_id=r.case_id, subject=r.subject_n, img_name=r.img_name, img_split=r.img_split,
                      r_max=rmax, match_case=match, same_image=bool(rmax >= SAME_IMG_R),
                      match_role=ROLE.get(match) if rmax >= SAME_IMG_R else None,
                      n_same_subject_train=len(same_subj),
                      same_subject_roles=",".join(sorted({ROLE.get(n, "?") for n in same_subj}))))
AUD = pd.DataFrame(_rows)
T.write_once_csv(AUD, OUT / "segmentation_audit.csv")

_exposed = AUD.same_image | (AUD.n_same_subject_train > 0)
if PROV["missing"] or not TRAIN_IMGS:
    VERDICT = "thieu_provenance"
elif _exposed.any():
    VERDICT = "phoi_nhiem"
else:
    VERDICT = "sach"
SUMMARY = dict(PROV, verdict=VERDICT, n=len(AUD), n_same_image=int(AUD.same_image.sum()),
               same_image_roles=AUD.match_role.value_counts().to_dict(),
               n_same_subject=int((AUD.n_same_subject_train > 0).sum()), same_image_r=SAME_IMG_R,
               r_max_not_same=float(AUD.loc[~AUD.same_image, "r_max"].max()) if (~AUD.same_image).any() else None,
               subinfo=[str(p) for p in SUBINFO], git=GIT_SHA)
if not (OUT / "segmentation_audit.json").exists():
    T.write_once_json(SUMMARY, OUT / "segmentation_audit.json")
print(json.dumps({k: v for k, v in SUMMARY.items() if k != "checkpoints"}, indent=1))
print("\nKET LUAN:", {"sach": "SACH - duoc tiep tuc buoc 2 (segmentation 96 anh bang checkpoint nay)",
                      "phoi_nhiem": "PHOI NHIEM - KHONG duoc goi la danh gia tren ca chua thay; can checkpoint sach",
                      "thieu_provenance": "THIEU PROVENANCE - sua SEG_* cho dung mo hinh da dung roi chay lai"}[VERDICT])

## 3) Các bước sau — **chưa triển khai**

Theo `docs/ke_hoach_m3t_cls.md` mục 5b, chỉ làm khi mục 2 kết luận **sạch** và S9 đã có `m3t_cls.csv`:

2. Segmentation 96 ảnh bằng checkpoint đã khóa; kiểm đủ mask, nhãn, hình học, QC bằng ảnh (mask AI, không thay GT).
3. 15 biomarker S3 + 55 biomarker S6 bằng đúng định nghĩa của cohort phát triển (`bsc/biomarkers.py`), bảng 96 dòng
   riêng + provenance.
4. Ghép CLS theo `case_id` (`validate="one_to_one"`), đủ 96 ca, không imputation che ca thiếu.
5. Fit lại đúng hai nhánh B chính trên 1229 ca, đánh giá holdout **một lần**; không dùng holdout để chọn gì.
6. Radiomics trên holdout là tùy chọn phụ.

Tiêu chí đăng ký: khi ΔQWK chính trên S7 > 0 thì ΔQWK holdout > 0 là **cùng chiều** — bằng chứng hỗ trợ yếu ở n = 96,
không nâng mức kết luận của S7.